In [ ]:
# This is useful if you are working with a dev install of asf_search and experimenting with changes to the codebase
%load_ext autoreload
%autoreload 2

## Create `SBASNetwork`s

### Search for a geographic reference scene

In [ ]:
import asf_search as asf

asf.constants.INTERNAL.CMR_TIMEOUT = 600

results = asf.product_search('S1A_IW_SLC__1SDV_20220215T225119_20220215T225146_041930_04FE2E_9252-SLC')
reference = results[0]
reference

### Create an `SBASNetwork`, defining:
- temporal bounds
- seasonal bounds
- geographic reference scene
- perpendicular baseline
- in-season temporal baseline (temporal baeline, excluding those for annual or multiannual seasonal bridge pairs)
- target bridge date (target date from which to bridge seasonal gaps)
- bridge year threshold (number of years across which to create bridge pairs)

In [ ]:
from datetime import datetime, date
import pandas as pd

def get_julian_season(season) -> tuple[int,int]:
    season_start_ts = pd.Timestamp(
        datetime.strptime(f"{season[0]}-0001", "%m-%d-%Y"), tz="UTC"
        )
    season_start_day = season_start_ts.timetuple().tm_yday
    season_end_ts = pd.Timestamp(
        datetime.strptime(f"{season[1]}-0001", "%m-%d-%Y"), tz="UTC"
    )
    season_end_day = season_end_ts.timetuple().tm_yday
    return (season_start_day, season_end_day)

season = ("1-1", "6-25")

opts = asf.ASFSearchOptions(
    **{
        "start": '2023-01-01', 
        "end": '2025-10-02', 
        "season": get_julian_season(season)
        }
)

sbas = asf.SBASNetwork(
    geo_reference = reference,
    perpendicular_baseline=100, 
    inseason_temporal_baseline=24,
    bridge_target_date='3-1',
    bridge_year_threshold=2,
    opts=opts,
    allow_missing_state_vectors=True)

print(len(sbas.full_stack))

### Alternatively, create an `SBASNetwork` by passing `baseline.stack` search results instead of a geographic reference product.

This allows you to pass stack search results that you have potentially altered in some way, as opposed to allowing the SBASNetwork class to a perform generic stack search on a geographic reference product.

In [ ]:


stack_search_results = reference.stack(opts=opts)
altered_stack_search_results = stack_search_results[1:-1] # remove some products from the search results

sbas_from_search_results = asf.SBASNetwork.from_search_results(
    altered_stack_search_results,
    perpendicular_baseline=200, 
    inseason_temporal_baseline=24,
    bridge_target_date='3-1',
    bridge_year_threshold=2,
    opts=opts)

print(len(sbas_from_search_results.full_stack))

### When creating an `SBASNetwork`, opt to accept or reject `Pairs` containing SLCs with missing state vectors

Some SLCs may be archived with missing state vectors, preventing the calculation of perpendicular baselines between reference and secondary scenes.

When building an `SBASNetwork`, you can opt to include or exclude Pairs with `None` baselines using the `allow_missing_state_vectors` argument.

Below, we intentionally delete an SLC's state vectors.

In [ ]:
stack_search_results = reference.stack(opts=opts)

# Set the SLC's state vector position paramaters to None
for key in stack_search_results[2].baseline["stateVectors"]["positions"].keys():
    stack_search_results[2].baseline["stateVectors"]["positions"][key] = None

stack_search_results[2].baseline["stateVectors"]["positions"]

### Create an `SBASNetwork`, excluding any Pairs containing SLCs with missing state vectors

In [ ]:
# The SBASNetwork class constructor defaults to excluding Pairs with missing state vectors
missing_state_vectors_not_allowed_sbas = asf.SBASNetwork.from_search_results(
    stack_search_results,
    perpendicular_baseline=200, 
    inseason_temporal_baseline=24,
    bridge_target_date='3-1',
    bridge_year_threshold=2,
    opts=opts)

print(f"The SBASNetwork's full_stack contains {len(missing_state_vectors_not_allowed_sbas.full_stack)} Pairs\n")
none_perp_baseline = [pair for pair in missing_state_vectors_not_allowed_sbas.full_stack if pair.perpendicular_baseline is None]
print(f"The SBASNetwork contains {len(none_perp_baseline)} Pairs having perpendicular baselines with a value of None")

### Create an `SBASNetwork` from the same search results, using the `allow_missing_state_vectors` argument to allow Pairs missing state vectors to be included

In [ ]:
missing_state_vectors_allowed_sbas = asf.SBASNetwork.from_search_results(
    stack_search_results,
    perpendicular_baseline=100, 
    inseason_temporal_baseline=24,
    bridge_target_date='3-1',
    bridge_year_threshold=2,
    opts=opts,
    allow_missing_state_vectors=True)

print(f"The SBASNetwork's full_stack contains {len(missing_state_vectors_allowed_sbas.full_stack)} Pairs\n")
none_perp_baseline = [pair for pair in missing_state_vectors_allowed_sbas.full_stack if pair.perpendicular_baseline is None]
print(f"The SBASNetwork contains {len(none_perp_baseline)} Pairs having perpendicular baselines with a value of None")

<hr>

## Plot SBASNetworks

### Plot the first SBASNetwork we created

If the network is disconnected, each connected substack will be plotted in a different color.

In [ ]:
sbas.plot()

### Add an automatically filtered Pair back to the `SBASNetwork` to manually connect one of the disconnected sub-networks

Use the `SBASNetwork.get_pair_from_dates()` method to identify a Pair to return to the network



In [ ]:
removed_pair_to_return_to_network = sbas.get_pair_from_dates(sbas.remove_list, date(2023,1,17), date(2024,6,16))
print(removed_pair_to_return_to_network)
print(f"Pair Dates: {removed_pair_to_return_to_network.ref_time.date()} - {removed_pair_to_return_to_network.sec_time.date()}")

sbas.add_pairs(removed_pair_to_return_to_network)

### Re-plot the the `SBASNetwork`

Note that the selected Pair has beeed back to the network, connecting a previously disconnected sub-network.

In [ ]:
sbas.plot()

### Remove pairs from the network

In [ ]:
pair1_to_remove = sbas.get_pair_from_dates(sbas.subset_stack, date(2025,1,6), date(2025,1,18))
pair2_to_remove = sbas.get_pair_from_dates(sbas.subset_stack, date(2025,1,6), date(2025,1,30))
print(f"{pair1_to_remove}, {pair2_to_remove}")
print(f"Pair 1 Dates: {pair1_to_remove.ref_time.date()} - {pair1_to_remove.sec_time.date()}")
print(f"Pair 2 Dates: {pair2_to_remove.ref_time.date()} - {pair2_to_remove.sec_time.date()}")

sbas.remove_pairs([pair1_to_remove, pair2_to_remove])

### Re-plot the network after removing 2 Pairs

Note that after removing both Pairs, the date 2025-01-06 is no longer included in the network at all and its vertex on the plot is red.

In [ ]:
sbas.plot()

### You can also add Pairs that were not included in the initial search and are not included in `SBASNetwork.remove_list` at all

In [ ]:
existing_pair = sbas.get_pair_from_dates(sbas.subset_stack, date(2025,3,7), date(2025,3,19))
existing_ref_scene = existing_pair.ref

results = asf.product_search('S1A_IW_SLC__1SDV_20260401T105744_20260401T105802_063885_0808EC_7B1B-SLC')
new_sec_scene = results[0]

new_pair = asf.Pair(existing_ref_scene, new_sec_scene)
print(new_pair)
print(f"Pair Dates: {new_pair.ref_time.date()} - {new_pair.sec_time.date()}")

sbas.add_pairs(new_pair)

### Re-plot the network after adding a Pair that was not in the scope of the original `SBASNetwork`'s time bounds

Note that there is now a pair with a secondary scene date in 2026. While this new pair has been manually added to the network, its perpendiccular baseline was not determined during the initial search and is therefore set to `None`. 

In [ ]:
sbas.plot()

### Plot a different network

By default `SBASNetwork.plot()` plots all connected sub-networks (`sbas.connected_substacks`)

However, `SBASNetwork` contains multiple networks that you can plot:
- `SBASNetwork.full_stack`: an SBAS network filtered by baselines, and including every possible bridge pair within SBASNetwork.bridge_year_threshold
- `SBASNetwork.subset_stack`: a possibly disconnected SBAS network filtered by baselines, with additional bridge pair filtering applied
- `SBASNetwork.connected_substacks`: SBASNetwork.subset_stack, broken up into a list of connected sub-networks

The code cell below plots `sbas.full_stack`


In [ ]:
sbas.plot(sbas.full_stack)

### Plot the SBASNetwork that contains missing state vectors, resulting in perpendicular baselines with values of None

Note that perpendicular baselines with None values are plotted with values of 0.0 m.

To view the perpendicular baselines of None on the plot, hover over the edges representing the following date pairs:
- 2023/01/05 - 2023/01/29
- 2023/01/17 - 2023/01/29
- 2023/01/29 - 2023/02/10
- 2023/01/29 - 2023/02/22

In [ ]:
missing_state_vectors_allowed_sbas.plot()

<hr>

## View the SLC IDs of an `SBASNetwork`'s `Pairs`

### View `SBASNetwork.subset_stack`'s scene ID's 

When ordering data from ASF [HyP3](https://hyp3-docs.asf.alaska.edu/hyp3-docs/about/) or [HyP3+](https://hyp3-docs.asf.alaska.edu/hyp3-docs/about/hyp3_plus/) On-Demand Processing, it is helpful to provide an `SBASNetwork` as a list of scene IDs.

The `SBASNetwork.get_scene_ids()` method (inherited from the `Stack` class) offers this functionality.

In [ ]:
largest_fully_connected_subset_stack_ids = sbas.get_scene_ids()
largest_fully_connected_subset_stack_ids

### You can view the scene IDs of any stack_list contained in a `SBASNetwork` object, including the complete `SBASNetwork.full_stack`, `SBASNetwork.remove_list`, the possibly disconnected `SBASNetwork.subset_stack`, or any of the stack_lists in `SBASNetwork.connected_substacks`

In [ ]:
remove_list_scene_ids = sbas.get_scene_ids(sbas.remove_list)
remove_list_scene_ids